# Microsoft Foundry ಜೊತೆಗೆ ಮಾದರಿಯನ್ನು ಫೈನ್-ಟ್ಯೂನ್ ಮಾಡುವುದು

ಈ ನೋಟ್‌ಬುಕ್ ಪ್ರಸ್ತುತ [Microsoft Foundry ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ವರ್ಕ್‌ಫ್ಲೋ](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst) ಅನ್ನು ಅನುಸರಿಸುತ್ತದೆ. ಇದು ನಿಮ್ಮ Foundry ಸಂಪನ್ಮೂಲದ `/openai/v1/` ಎಂಡ್‌ಪಾಯಿಂಟ್‌ಗೆ ಸೂಚಿಸುವ **OpenAI Python SDK (v1)** ಬಳಕೆ ಮಾಡುತ್ತದೆ, ಆದ್ದರಿಂದ ಒಂದೇ ಕೋಡ್ OpenAI ಪ್ಲಾಟ್‌ಫಾರ್ಮ್‌ಗೆ ಸಹ ಆ ಮೂಲಕ ಮೈಚಾರಿಕೆಯನ್ನು ಬದಲಾಯಿಸಿ ರನ್ ಆಗುತ್ತದೆ.

> **ಅಗತ್ಯಗಳು**
> - [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) ಯೋಜನೆ, **Foundry ಮಾಲೀಕ** ಹುದ್ದೆಯೊಂದಿಗೆ (ಫೈನ್-ಟ್ಯೂನ್ ಮತ್ತು ನಿಯೋಜನೆಗೆ ಅಗತ್ಯ).
> - `pip install "openai>=1.0" python-dotenv`
> - `AZURE_OPENAI_ENDPOINT` ಮತ್ತು `AZURE_OPENAI_API_KEY` ಹೊಂದಿರುವ `.env` ಫೈಲ್ (ಕೊರ್ಸ್ ಸೆಟ್-ಅಪ್ ಮಾರ್ಗದರ್ಶಿಯನ್ನು ನೋಡಿ: [course setup guide](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)).
> - ಪ್ರಸ್ತುತ ಬೆಂಬಲಿಸಲ್ಪಟ್ಟ ಮೂಲ ಮಾದರಿ, ಉದಾಹರಣೆಗೆ `gpt-4.1-mini` (ಫೈನ್-ಟ್ಯೂನಬಲ್ ಮಾದರಿಗಳ ಪಟ್ಟಿಯನ್ನು ನೋಡಿ: https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models).

ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಎಂದರೆ, ನಿಮ್ಮ ಕಾರ್ಯಕ್ಕೆ ಸಂಬಂಧಿಸಿದ ಹೆಚ್ಚುವರಿ ಉದಾಹರಣೆಗಳ ಮೇಲೆ ಮರುಶಿಕ್ಷಣ ನೀಡಿ ಆಧಾರ ಮಾದರಿಯನ್ನು ಸುಧಾರಿಸುವುದು. _few-shot learning_ ಮತ್ತು _retrieval-augmented generation_ ಮುಂತಾದ ಪ್ರಾಂಪ್ಟ್ ಎಂಜಿನಿಯರಿಂಗ್ ತಂತ್ರಗಳು ಸಂಬಂಧಿತ ಡೇಟಾವನ್ನು ಪ್ರಾಂಪ್ಟ್‌ಗೆ ಸೇರಿಸುತ್ತವೆ, ಆದರೆ ಅವು ಮಾದರಿಯ ಕಂಟೆಕ್ಸ್ಟ್ ವಿಂಡೋ ಮೂಲಕ ಮಿತಿಯಾಗಿರುತ್ತವೆ.

ಫೈನ್-ಟ್ಯೂನಿಂಗ್‌ ಮೂಲಕ ನೀವು ಸ್ವತಃ ಮಾದರಿಯನ್ನು ಮರುಶಿಕ್ಷಣ ನೀಡುತ್ತೀರಿ (ಕಂಟೆಕ್ಸ್ಟ್ ವಿಂಡೋ ಒಳಗೆ ಬರುವುದಕ್ಕಿಂತ ಬಹಳ ಹೆಚ್ಚು ಉದಾಹರಣೆಗಳೊಂದಿಗೆ) ಮತ್ತು ಒಂದು _ಪರಿವರ್ತಿತ_ ಆವೃತ್ತಿಯನ್ನು ನಿಯೋಜಿಸುತ್ತೀರಿ, ಅದು ಇನ್ಫರೆನ್ಸ್ ಸಮಯದಲ್ಲಿ ಆ ಉದಾಹರಣೆಗಳ ಅಗತ್ಯವಿಲ್ಲ. ಇದರಿಂದ ಗುಣಮಟ್ಟ ಸುಧಾರಿಸುತ್ತದೆ, ಕಂಟೆಕ್ಸ್ಟ್ ವಿಂಡೋ ಮುಕ್ತವಾಗುತ್ತದೆ, ಹಾಗೂ ಪ್ರಾಂಪ್ಟ್‌ಗಳು ಚಿಕ್ಕದಾಗುವುದರಿಂದ ವೆಚ್ಚ ಮತ್ತು ವಿಳಂಬ ಕಡಿಮೆಯಾಗಬಹುದು. ಫೌಂಡ್ರಿ ಹೀಗಾಗಿ **LoRA (low-rank adaptation)** ಅನ್ನು ಪರಿಣಾಮಕಾರಿ ತರಬೇತಿಯಿಗಾಗಿ ಬಳಕೆ ಮಾಡುತ್ತದೆ.

ಫೌಂಡ್ರಿ ಮೂರು ತಂತ್ರಗಳನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ: **ಸೂಪರ್ವೈಸ್ಡ್ ಫೈನ್-ಟ್ಯೂನಿಂಗ್ (SFT)** - ಈ ಪಾಠದಲ್ಲಿ ಬಳಸಲಾಗಿದೆ - ಜೊತೆಗೆ **DPO** (ಆಸಕ್ತಿಯ ಸರಿಹೊಂದಿಸುವಿಕೆ) ಮತ್ತು **RFT** (ಬಲವರ್ಧಿತ ಫೈನ್-ಟ್ಯೂನಿಂಗ್). ವರ್ಕ್‌ಫ್ಲೋ ನಾಲ್ಕು ಹಂತಗಳಿವೆ:

1. ನಿಮ್ಮ ತರಬೇತಿ **ಮತ್ತು ಮಾನ್ಯತೆ** ಡೇಟಾವನ್ನು ಸಿದ್ಧಪಡಿಸಿ ಮತ್ತು ಅಪ್‌ಲೋಡ್ ಮಾಡಿ.
2. ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಮಾದರಿಯನ್ನು ಸೃಷ್ಟಿಸಲು ತರಬೇತಿ ಕೆಲಸವನ್ನು ಚಾಲನೆ ಮಾಡಿರಿ.
3. ಜಾಬ್ ನಿಗಾದಾನ ಮಾಡಿ, ಮೆಟ್ರಿಕ್ಸ್ ಪರಿಶೀಲಿಸಿ ಮತ್ತು ಒಂದು ಚೆಕ್ಪಾಯಿಂಟ್ ಆಯ್ಕೆಮಾಡಿ.
4. ಫೈನ್-ಟ್ಯೂನ್ ಮಾಡಿದ ಮಾದರಿಯನ್ನು ನಿಯೋಜಿಸಿ ಮತ್ತು ಇನ್ಫರೆನ್ಸ್ ಗೆ ಬಳಸಿಕೊಳ್ಳಿ.

ಈ ಪಾಠದಲ್ಲಿ ನಾವು SFT ಜೊತೆ `gpt-4.1-mini` ಅನ್ನು ಫೈನ್-ಟ್ಯೂನ್ ಮಾಡಿ, ಮಧ್ಯಯುಗದ ಪದ್ಯದ ಶೈಲಿಯಲ್ಲಿ ಪಿರಿಯೋಡಿಕ್ ಟೇಬಲ್ ಕುರಿತು ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸುವ ಚಾಟ್‌ಬಾಟ್ ಅನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.

---


### ಹಂತ 1.1: ನಿಮ್ಮ ಡೇಟಾಸೆಟ್ ತಯಾರಿಸಿ

ಲಿಮೇರಿಕ್ ಮೂಲಕ ಒಂದು ಅಂಶದ ಬಗ್ಗೆ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸುವ ಮೂಲಕ ಪದಾರ್ಥಗಳ ನಿಯಮಿತ ಟೇಬಲನ್ನು ನಿಮಗೆ ಅರ್ಥಮಾಡಿಕೊಡಲು ಸಹಾಯಮಾಡುವ ಚಾಟ್‌ಬಾಟ್ ನಿರ್ಮಿಸೋಣ. _ಈ_ ಸರಳ ತರಗತಿಯಲ್ಲಿ, ನಮಗೆ ಕೇವಲ ಮಾದರಿಯನ್ನು ತರಬೇತಿಗಾಗಿ ಕಂಡುಹಿಡಿದ ಕೆಲವು ಪ್ರತಿಕ್ರಿಯೆ ಉದಾಹರಣೆಗಳೊಂದಿಗೆ ಒಂದು ಡೇಟಾಸೆಟ್ ಸೃಷ್ಟಿಸುವದು. ನೈಜ-ಜೀವನವಧ್ಯದ ಬಳಕೆಯಲ್ಲಿ, ನೀವು ಹೆಚ್ಚಿನ ಉದಾಹರಣೆಗಳೊಂದಿಗೆ ಒಂದು ಡೇಟಾಸೆಟ್ ವನ್ನು ರಚಿಸಬೇಕಾಗುತ್ತದೆ. ನಿಮ್ಮ ಅಪ್ಲಿಕೇಶನ್ ಡೊಮೇನ್ಗಾಗಿ ತೆರೆಯಲಾದ ಡೇಟಾಸೆಟ್ (open dataset) ಇದೆಯಾದರೆ ಅದನ್ನು ಬಳಸಿಕೊಂಡು, ಫೈನ್-ಟ್ಯೂನಿಂಗ್‌ಗೆ ಅನ್ವಯವಾಗುವಂತೆ ಮರುರೂಪಿಸಬಹುದು.

ನಾವು `gpt-4.1-mini` ಗೆ ಕೇಂದ್ರೀಕರಿಸುತ್ತಿದ್ದಾಗ ಮತ್ತು ಒಂದೇ ಬಾರಿ ಉತ್ತರ (ಚಾಟ್ ಪೂರ್ಣತೆ) ಬೇಕಾದ್ದರಿಂದ, [ಈ ಸೂಚಿಸಿದ ಫಾರ್ಮ್ಯಾಟ್](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ಬಳಸಿ ಉದಾಹರಣೆಗಳನ್ನು ರಚಿಸಬಹುದು ಇದು OpenAI ಚಾಟ್ ಪೂರ್ಣತೆ ಅವಶ್ಯಕತೆಗಳಿಗೆ ತಕ್ಕಂತೆ ಇದೆ. ನೀವು ಬಹುಮಟ್ಟದ ಸಂವಹನ ವಿಷಯವನ್ನು ನಿರೀಕ್ಷಿಸುವಿದ್ದರೆ, ನೀವು [ಬಹು-ತಟ(ಮಲ್ಟಿ) ಚಾಟ್ ಉದಾಹರಣೆ ಫಾರ್ಮ್ಯಾಟ್](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) ಬಳಸುತ್ತೀರಿ ಇದರಲ್ಲಿ fine-tuning ಪ್ರಕ್ರಿಯೆಯಲ್ಲಿ ಯಾವ ಸಂದೇಶಗಳನ್ನು ಉಪಯೋಗಿಸಬೇಕೆಂದು ಸೂಚಿಸುವ `weight` ಪರಿಮಾಣವೂ ಇದೆ.

ನಾವು ಇಲ್ಲಿ ನಮ್ಮ ತರಗತಿಯಿಗಾಗಿ ಸರಳ ಒಂದೇ-ತಟ ಫಾರ್ಮ್ಯಾಟ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ. ಡೇಟಾ [jsonl ಫಾರ್ಮ್ಯಾಟ್](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) ನಲ್ಲಿ ಇದೆ, ಪ್ರತಿಯೊಂದು ಸಾಲಿನಲ್ಲಿ ಒಂದು ದಾಖಲೆ, ಪ್ರತಿ ದಾಖಲೆ JSON-ರೂಪದಲ್ಲಿ ಪ್ರತಿನಿಧಿಸಲಾಗುತ್ತದೆ. ಕೆಳಗಿನ ತುಣುಕು ಎರಡು ದಾಖಲೆಗಳನ್ನು ಮಾದರಿಯಾಗಿ ತೋರಿಸುತ್ತದೆ - ಸಂಪೂರ್ಣ ಮಾದರಿ ಸೆಟ್ (10 ಉದಾಹರಣೆಗಳು)ಗಾಗಿ [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) ನೋಡಿ ನಾವು ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ತರಗತಿಗೆ ಬಳಸೋದು. **ಟಿಪ್ಪಣಿ:** ಪ್ರತಿಯೊಂದು ದಾಖಲೆ ಒಂದು ಸಾಲಿನಲ್ಲಿ _ತಯಾರಿಸಬೇಕಾಗುತ್ತದೆ_ (ಸಾಮಾನ್ಯ JSON ಫೈಲ್ ಅನ್ನುಂತಲ್ಲ, ಯಾವುದೇ ಸಾಲು ವಿಭಜನೆ ಇಲ್ಲದೆ)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

ನೈಜ-ಜೀವನದ ಬಳಕೆಯಲ್ಲಿ ಉತ್ತಮ ಫಲಿತಾಂಶಕ್ಕಾಗಿ ದೊಡ್ಡ ಪ್ರಮಾಣದ ಉದಾಹರಣೆಗಳ ಸೆಟ್ ಬೇಕಾಗುತ್ತದೆ - ಇದಕ್ಕೆ ತೀರಾ ಉತ್ತಮ ಉತ್ತರಗಳ ಗುಣಮಟ್ಟ ಮತ್ತು ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಸಮಯ/ಖರ್ಚಿನ ನಡುವೆ ವ್ಯವಹಾರ ಇರುತ್ತದೆ. ನಾವು ಸಣ್ಣ ಪ್ರಮಾಣದ ಸೆಟ್ ಅನ್ನು ಬಳಸಿ ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಪ್ರಕ್ರಿಯೆಯನ್ನು ತ್ವರಿತವಾಗಿ ಪೂರ್ಣಗೊಳಿಸಿ ವೀಕ್ಷಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದೇವೆ. ಹೆಚ್ಚು ಸಂಕೀರ್ಣವಾದ ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ತರಗತಿಯಿಗಾಗಿ [ಈ OpenAI ಕುಕ್‌ಬುಕ್ ಉದಾಹರಣೆಯನ್ನು](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) ನೋಡಿ.


---

### ಹಂತ 1.2: ನಿಮ್ಮ ಡೇಟಾಸೆಟ್ ಅನ್ನು ಅಪ್‌ಲೋಡ್ ಮಾಡಿ

Files API (`purpose="fine-tune"`) ಬಳಸಿ ನಿಮ್ಮ ತರಬೇತಿ ಮತ್ತು ಮಾನ್ಯತಾ ಫೈಲ್‌ಗಳನ್ನು Foundry ಗೆ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ. **ಮಾನ್ಯತಾ ಫೈಲ್** ಒದಗಿಸುವುದು Foundry ಗೆ ಮಾನ್ಯತಾ ನಷ್ಟವನ್ನು ವರದಿ ಮಾಡಲು ಅನುವು ಮಾಡಿಕೊಡುತ್ತದೆ, ಇದರಿಂದ ನೀವು ಅತಿರೇಕ ಸೋಧನೆಯನ್ನು ಗಮನಿಸಬಹುದು.

ಕೆಳಗಿನ ಕೋಡ್ ಚಲಾಯಿಸುವ ಮೊದಲು, ಕೆಳಗಿನವುಗಳನ್ನು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ:
 - SDK ಅನ್ನು ಸ್ಥಾಪಿಸಲಾಗಿದೆ: `pip install "openai>=1.0" python-dotenv`
 - `AZURE_OPENAI_ENDPOINT` ಮತ್ತು `AZURE_OPENAI_API_KEY` ಹೊಂದಿರುವ `.env` ಫೈಲ್ ಅನ್ನು ರಚಿಸಲಾಗಿದೆ

ಈ ಕೋಡ್ Foundry ಸಂಪನ್ಮೂಲದ `/openai/v1/` ಎಂಡ್ಪಾಯಿಂಟ್ ಕಡೆಗೆ ಪಾಯಿಂಟ್ ಮಾಡಿದ OpenAI ಕ್ಲೈಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುತ್ತದೆ, ನಂತರ ಎರಡು ಫೈಲ್‌ಗಳನ್ನೂ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ ಅವುಗಳ ID ಗಳನ್ನೂ ಮುದ್ರಿಸುತ್ತದೆ.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### ಹಂತ 2.1: SDK ಬಳಸಿ ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಕೆಲಸವನ್ನು ರಚಿಸಿ


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### ಹಂತ 2.2: ಕೆಲಸದ ಸ್ಥಿತಿಯನ್ನು ಪರಿಶೀಲಿಸಿ

ನೀವು `client.fine_tuning.jobs` API ನೊಂದಿಗೆ ಮಾಡಬಹುದಾದ ಕೆಲವು ವಿಷಯಗಳು ಇವೆ:
- `client.fine_tuning.jobs.list(limit=<n>)` - ಕೊನೆಯ n ಫೈನ್-ಟ್ಯೂನಿಂಗ್ ಕೆಲಸಗಳನ್ನು ಪಟ್ಟಿ ಮಾಡಿ
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ವಿಶೇಷ ಕೆಲಸದ ವಿವರಗಳನ್ನು ಪಡೆಯಿರಿ
- `client.fine_tuning.jobs.cancel(<job_id>)` - ಕೆಲಸವನ್ನು ರದ್ಧುಮಾಡಿ
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - ಕೆಲಸದಿಂದ ಈವೆಂಟ್ ಗಳು ಪಟ್ಟಿ ಮಾಡಿ
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - വിനಿಯೋಗಿಸಬಹುದಾದ ಚೆಕ್ ಪಾಯಿಂಟ್ ಗಳನ್ನು ಪಟ್ಟಿ ಮಾಡಿ (ಕೊನೆಯ ಕೆಲವು ಪೀರಿoಗಳು)

ಕೆಲಸ ಪ್ರಾರಂಭವಾದಾಗ, Foundry ಮೊದಲು _ಪ್ರಶಿಕ್ಷಣ ಕಡತವನ್ನು ಮಾನ್ಯಗೊಳಿಸುತ್ತದೆ_ ಡೇಟಾ ಸರಿಯಾದ ಸ್ವರೂಪದಲ್ಲಿದೆ ಎಂದು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಲು. ನಂತರ ತರಬೇತಿ ನಡೆಯುತ್ತದೆ ಮತ್ತು ಮಾದರಿ ಮತ್ತು ಡೇಟಾಸೆಟ್ ಗಾತ್ರದ ಮೇಲೆ ಅವಲಂಬಿಸಿ ನಿಮಿಷಗಳಿಂದ ಗಂಟೆಗಳವರೆಗೆ ತೆಗೆದುಕೊಳ್ಳಬಹುದು.


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### ಹಂತ 2.3: ಪ್ರಗತಿಯನ್ನು ಕೈಗೊಳ್ಳಲು ಘಟನೆಗಳನ್ನು ಹತ್ತಿರದಿಂದ ನೋಡಿಕೊಳ್ಳಿ


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### ಹಂತ 2.4: ಫೌಂಡ್ರಿ ಪೋರ್ಟಲ್‌ನಲ್ಲಿ ಸ್ಥಿತಿ, ಮೆಟ್ರಿಕ್ಸ್ ಮತ್ತು ಚೆಕ್ ಪಾಯಿಂಟ್‌ಗಳನ್ನು ಪರಿಶೀಲಿಸಿ


ನೀವು ನಿಮ್ಮ ಕೆಲಸವನ್ನು [Microsoft Foundry ಪೋರ್ಟಲ್](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) ನಲ್ಲಿ **Build > Fine-tuning** ಅಡಿಯಲ್ಲಿ ಸಹ ಟ್ರ್ಯಾಕ್ ಮಾಡಬಹುದು. ಅದರ ಸ್ಥಿತಿಯನ್ನು, ತರಬೇತಿ ಘಟನೆಗಳನ್ನು, ಹೈಪರ್‌ಪ್ಯಾರಾಮೀಟರ್‌ಗಳನ್ನು ಮತ್ತು ಮ್ಯೆಟ್ರಿಕ್‌ಗಳನ್ನು ನೋಡಲು ನಿಮ್ಮ ಕೆಲಸವನ್ನು ಆಯ್ಕೆಮಾಡಿ. ಈ ಮ್ಯೆಟ್ರಿಕ್‌ಗಳನ್ನು ಗಮನಿಸಿ:

- `train_loss` ಮತ್ತು `full_valid_loss` - ಕಾಲಕ್ರಮೇಣ ಕಡಿಮೆಯಾಗಬೇಕು.
- `train_mean_token_accuracy` ಮತ್ತು `full_valid_mean_token_accuracy` - ಹೆಚ್ಚಾಗಬೇಕು.

ತರಬೇತಿಗೊಡನೆಯೂ ಮಾನ್ಯತೆ ವಕ್ರಗಳು ವಿಭಿನ್ನವಾಗಿದ್ದರೆ, ನೀವು ದೊಡ್ಡ ಮಟ್ಟದಲ್ಲಿ ಅತಿವ್ಯವಸ್ಥೆ ಮಾಡುತ್ತಿದ್ದೀರಿ ಎಂದು ಅನುಮಾನಿಸಬಹುದು - ಕಡಿಮೆ ಎಪೋಚ್ಗಳನ್ನು ಅಥವಾ ಚಿಕ್ಕ ಕಲಿಕೆ-ದರ ಗುಣಕವನ್ನು ಪ್ರಯತ್ನಿಸಿ. ಕೆಳಗಿನ ಚಿತ್ರಣವು ನೀವು ಕಾಣಲಿರುವ ಸ್ಥಿತಿ, ಸಂದೇಶ ಮತ್ತು ಮ್ಯೆಟ್ರಿಕ್ ಫಲಕಗಳ ಪ್ರಕಾರವನ್ನು ತೋರಿಸುತ್ತದೆ (ನಿಖರ UI ಪ್ರೊವೈಡರ್ ಅನುಸಾರ ಬದಲಾಗಬಹುದು).

![Fine-tuning job status](../../../../../translated_images/kn/fine-tuned-model-status.563271727bf7bfba.webp)


ನೀವು ಕೆಳಗಡೆ ಸ್ಕ್ರೋಲ್ ಮಾಡುವ ಮೂಲಕ ದೃಶ್ಯ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್‌ನಲ್ಲಿ ಸ್ಥಿತಿ ಸಂದೇಶಗಳು ಮತ್ತು ಮೆಟ್ರಿಕ್‌ಗಳನ್ನು ಕೂಡ ನೋಡಬಹುದು, ಹಾಗೆ ತೋರಿಸಲಾಗಿದೆ:

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/kn/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/kn/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### ಹಂತ 3.1: ಸೂಕ್ಷ್ಮವಾಗಿ ತರಬೇತಿಗೊಂಡ ಮಾದರಿಯ ಐಡಿ ಪಡೆಯಿರಿ

ನೌಕರಿ ಯಶಸ್ವಿಯಾಗುವಾಗ, `response.fine_tuned_model` ನಿಮ್ಮ ಕಸ್ಟಮ್ ಮಾದರಿಯ ಐಡಿಯನ್ನು ಹೊಂದಿರುತ್ತದೆ (ಉದಾಹರಣೆಗೆ, `gpt-4.1-mini-2025-04-14.ft-...`). ಆಜೂರ್‌ನಲ್ಲಿ ನೀವು ಅದನ್ನು ಕರೆದೋಲು ಮುಂಚೆ ಆ ಮಾದರಿಯನ್ನು **ಪ್ರತಿಷ್ಠಾಪಿಸಬೇಕು**.


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### ಹಂತ 3.2: ಸೂಕ್ಷ್ಮ-ಸಂರಚಿತ ಮಾದರಿಯನ್ನು ನಿಯೋಜಿಸಿ

OpenAI ವೇದಿಕೆಯಂತೆ (ನೀವು ಸೂಕ್ಷ್ಮ-ಸಂರಚಿತ ಮಾದರಿ ಐಡಿಯನ್ನು ನಿರ್ವಹಿಸಬಹುದು), Microsoft Foundry ನಿಮಗೆ ಮೊದಲಿಗೆ ಮಾದರಿಯನ್ನು **ನಿಯೋಜಿಸಲು** ಅಗತ್ಯವಿದೆ. ಸುಲಭ ಮಾರ್ಗವೆಂದರೆ [Foundry ಪೋರ್ಟಲ್](https://ai.azure.com?WT.mc_id=academic-105485-koreyst): **Build > Fine-tuning** ಗೆ ಹೋಗಿ, ನಿಮ್ಮ ಮುಗಿದ ಕೆಲಸವನ್ನು തിരഞ്ഞെടಿಸಿ ಮತ್ತು **Deploy** ಆಯ್ಕೆಮಾಡಿ. ನಿಯೋಜನೆಯಿಗೆ ಹೆಸರು ನೀಡಿ (ಉದಾಹರಣೆಗೆ, `elements-limerick`) - ಆ ನಿಯೋಜನೆಯ ಹೆಸರು ನಿಮಗೆ ಕೋಡ್‌ನಿಂದ ಕರೆ ಮಾಡಲು ಬೇಕಾಗುತ್ತದೆ.

ನೀವು ನಿಯಂತ್ರಣ-ಪ್ಲೇನ REST/ARM API ಯೊಂದಿಗೆ ಪ್ರೋಗ್ರಾಮ್ಯಾಟಿಕಾಗಿ ನಿಯೋಜಿಸಬಹುದು; [deployment guide](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst) ನೋಡಿ. ನಿಯೋಜಿತ ಕಸ್ಟಮ್ ಮಾದರಿ ಘಂಟೆಗೆ ಬಿಲ್ ಮಾಡುತ್ತದೆ ಮತ್ತು 15 ದಿನಗಳ ನಂತರ ನಿಷ್ಕ್ರಿಯ ನಿಯೋಜನೆಯನ್ನು ತೆಗೆದುಹಾಕಲಾಗುತ್ತದೆ ಎಂಬುದನ್ನು ನೆನಪಿಡಿ.


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### ಹಂತ 3.3: ಫೌಂಡ್ರಿ ಪ್ಲೇಗ್ರೌಂಡಿನಲ್ಲಿ ನಿಮ್ಮ ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ಮಾದರಿಯನ್ನು ಪರೀಕ್ಷಿಸಿ

ನೀವು ಹೂಡಿಕೆಯಾದ ಮಾದರಿಯನ್ನು [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **ಪ್ಲೇಗ್ರೌಂಡ್** ನಲ್ಲಿ ಪರೀಕ್ಷಿಸಬಹುದು - ಮಾದರಿ ಡ್ರಾಪ್‌ಡೌನ್‌ನಿಂದ ನಿಮ್ಮ ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ನಿಯೋಜನೆಯನ್ನು ಆಯ್ಕೆಮಾಡಿ ಮತ್ತು ಪ್ರಾಂಪ್ಟ್ ಅನ್ನು ಪ್ರಯತ್ನಿಸಿ. ನೀವು ತರಬೇತುಗೊಂಡಿದ್ದ **ಆದ್ದೇ ವ್ಯವಸ್ಥೆ ಸಂದೇಶವನ್ನು** ಬಳಸಿ; ಬೇರೆ ವ್ಯವಸ್ಥೆ ಸಂದೇಶವೊಂದು ಪರಿಣಾಮವನ್ನು ಬದಲಾಯಿಸಬಹುದು.

> **ಟಿಪ್:** ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ಮಾದರಿಯನ್ನು ಮೂಲ `gpt-4.1-mini` ಜೊತೆಗೆ ಹೋಲಿಸಿ. ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ಮಾದರಿ ನಿಮ್ಮ ಉದಾಹರಣೆಗಳಿಂದ ಲಿಮೆರಿಕ್ ಸ್ವರೂಪದಲ್ಲಿ ಉತ್ತರಗಳನ್ನು ನೀಡುತ್ತದೆ, ಆದರೆ ಮೂಲ ಮಾದರಿ ಕೇವಲ ವ್ಯವಸ್ಥೆ ಪ್ರಾಂಪ್ಟ್ ಅನ್ನು ಅನುಸರಿಸುತ್ತದೆ.

**ಇದು ಪ್ರಕ್ರಿಯೆಯನ್ನು ತೋರಿಸಲು ಉದ್ದೇಶಿತ ಸರಳ ಉದಾಹರಣೆ, ನಿಜದ ಡೇಟಾಸೆಟ್ ಅಲ್ಲ.** ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು ನಿಜವಾದ ಡೇಟಾದ ಮೈಲೆ ಫೈನ್ಟ್ಯೂನ್ ಮಾಡುತ್ತೀರಿ (ಉದಾಹರಣೆಗೆ, ಗ್ರಾಹಕ ಸೇವೆಗಾಗಿ ಉತ್ಪನ್ನ ಪಟ್ಟಿಯು), ಅಲ್ಲಿ ಗುಣಮಟ್ಟದ ವ್ಯತ್ಯಾಸ ಸ್ಪಷ್ಟವಾಗಿರುತ್ತದೆ - ಮತ್ತು ಆ ಗುಣಮಟ್ಟವನ್ನು ಪ್ರಾಂಪ್ಟ್ ಎಂಜಿನಿಯರಿಂಗ್ ಮಾತ್ರದಿಂದ ಹೊಂದುವುದು ಪ್ರತಿಗೆ ಹಲವಾರು ಹೆಚ್ಚುವರಿ ಟೋಕನ್ಗಳನ್ನು ಬೆಲೆ ಇಟ್ಟಿರುತ್ತದೆ. ಇನ್ನಷ್ಟು ಸಂಪೂರ್ಣ ಉದಾಹರಣೆಗೆ, [OpenAI ಕ್ಯೂಕುಬುಕ್ ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ಮಾರ್ಗದರ್ಶಿ](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) ಮತ್ತು [ಫೌಂಡ್ರಿ ಸೂಕ್ಷ್ಮ-ಸಂಸ್ಥಿತ ಟ್ಯುಟೋರಿಯಲ್](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst) ನೋಡಿ.

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
